# Evaluating real agents with AuditKIT: `hf:`, Cohere and `agent:` endpoints

This notebook shows how to evaluate a **real** model's tool use with AuditKIT. Every answer here is generated live; nothing is precomputed.

| § | Endpoint | What you learn |
|---|---|---|
| 1 | `hf:` (Qwen3-1.7B) | Native tool calling through the chat template, and prompt mode for models whose template has no tools |
| 2 | `hf:` | Multi-turn: the model answers from a tool result already in the conversation |
| 3 | Cohere via `hf:` | Tiny Aya (prompt mode), and Command R7B (native tools, Cohere's own action format) |
| 4 | Cohere hosted API via `api:` | The same evaluation against Cohere's OpenAI-compatible endpoint |
| 5 | `agent:` (a deployed agent) | Evaluating an agent that runs its **own** tool loop behind HTTP, as an AgentTune-served agent would |
| 6 | `agent_eval` harness | AuditKIT drives the loop, executes the tools against a test double, and verifies the **final state** |

**Runtime:** a Colab **GPU** (a T4 is enough for §1, §2, §3a, §5 and §6). §3b (Command R7B) needs a 24 GB+ GPU. It also runs on a laptop GPU/MPS.

**Optional secrets:**
- `HF_TOKEN`, from an account that has accepted the licences for `CohereLabs/tiny-aya-global` / `CohereLabs/c4ai-command-r7b-12-2024` (§3);
- `CO_API_KEY` (§4);
- `GITHUB_TOKEN`, to install AuditKIT from the private repo.

Sections whose requirements are missing are skipped with a note.

> The outputs below are from a run on an Apple M3 Pro (MPS) with Qwen3-1.7B. The gated Cohere sections (3a, 3b) and the Cohere API section (4) were skipped there because no token was set. They run on Colab once you add the secrets.

In [1]:
# ---- settings ----
BRANCH = "test/agentic-rag-full-suite"          # an AuditKIT branch with #10 (hf: native tools, Cohere parsing)
MODEL = "Qwen/Qwen3-1.7B"                   # any hf: chat model whose template renders tools
TINY_AYA = "CohereLabs/tiny-aya-global"     # gated
COMMAND_R7B = "CohereLabs/c4ai-command-r7b-12-2024"   # gated, ~16 GB
COHERE_API_MODEL = "command-r7b-12-2024"
MAX_TOKENS = 768                            # Qwen3 thinks before it calls; give it room

In [2]:
# ---- install (Colab) / reuse (local checkout) ----
import os, subprocess, sys
try:
    import auditkit  # already importable (local checkout / installed)
except ImportError:
    from google.colab import userdata
    gh = userdata.get("GITHUB_TOKEN")
    for k in ("HF_TOKEN", "CO_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH,
                    f"https://{gh}@github.com/aryapratinavseth/AuditKIT-Internal.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", "AuditKIT-Internal[transformers,requests]"], check=True)
    sys.path.insert(0, "AuditKIT-Internal/src")
import auditkit as ak, torch
from auditkit.runspec import RunConfig
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("auditkit", getattr(ak, "__version__", "?"), "| device", DEVICE)

auditkit 0.3.0 | device mps


In [3]:
# ---- the tools and tasks shared by every section ----
import json
TOOLS = [
    {"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
]
W = lambda city: {"name": "get_weather", "arguments": {"city": city}}

# expected_tool_calls is a list of TURNS; a turn that is itself a list = calls made in parallel
TASKS = [
    ak.Sample(id="single",    input="What's the weather in Paris?",                   tools=TOOLS, expected_tool_calls=[W("Paris")]),
    ak.Sample(id="parallel",  input="What's the weather in Paris and in Rome? Call the tool for both cities at once.",
              tools=TOOLS, expected_tool_calls=[[W("Paris"), W("Rome")]]),
    ak.Sample(id="no_tool",   input="What is 2+2? Answer directly, do not use tools.", tools=TOOLS, expected_tool_calls=[]),
    ak.Sample(id="pick_tool", input="What time is it in Tokyo?",                      tools=TOOLS,
              expected_tool_calls=[{"name": "get_time", "arguments": {"city": "Tokyo"}}]),
]
SCORERS = lambda: [ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls(), ak.ToolCallValidity(),
                   ak.ToolPermission(), ak.AgentLoopDetection()]

def show(r, title):
    from auditkit.trace import parse_tool_calls
    print(f"=== {title} ===")
    print("headline:", {k: round(v, 3) for k, v in sorted(r.headline.items())})
    if r.errors:
        print("errors:", r.errors)
    for p in r.predictions:
        # the calls AuditKIT actually scored (from the reply text, or from an agent's returned transcript)
        scored = next((s["metadata"].get("predicted") for s in p.metadata.get("scores", [])
                       if s["name"] == "tool_call_precision" and "predicted" in (s.get("metadata") or {})), None)
        calls = scored if scored is not None else [(c.name, c.arguments) for c in parse_tool_calls(p.raw_output or "")]
        tail = (p.raw_output or "").split("</think>")[-1].strip()[:120]
        print(f"  {p.sample_id:10s} calls={calls}  reply_tail={tail!r}")
RESULTS = {}

## 1 · `hf:`, native tool calling

`ToolCallAdapter()` (native mode) passes the schemas as `tools=` to the model's **own chat template**, so the model sees them in the format it was trained on. The reply is the model's raw text; AuditKIT parses the calls from it (Hermes `<tool_call>` blocks for Qwen, action lists for Cohere) and scores them.

In [4]:
from auditkit.model.hf_gen import HFGenModel
qwen = HFGenModel(MODEL, device=DEVICE, name=f"hf:{MODEL}")
r = ak.evaluate(TASKS, model=qwen, adapter=ak.ToolCallAdapter(), scorers=SCORERS(), config=RunConfig(max_tokens=MAX_TOKENS))
show(r, "hf: native tools"); RESULTS["1 hf native"] = r.headline

=== hf: native tools ===
headline: {'agent_loop_detection': 1.0, 'parallel_detection': 1.0, 'parallel_precision': 1.0, 'parallel_recall': 1.0, 'tool_call_exact': 1.0, 'tool_call_f1': 1.0, 'tool_call_precision': 1.0, 'tool_call_recall': 1.0, 'tool_call_validity': 1.0, 'tool_permission': 1.0, 'trajectory_in_order': 1.0, 'trajectory_strict': 1.0}
  single     calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}}]  reply_tail='<tool_call>\n{"name": "get_weather", "arguments": {"city": "Paris"}}\n</tool_call>'
  parallel   calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}}, {'name': 'get_weather', 'arguments': {'city': 'Rome'}}]  reply_tail='<tool_call>\n{"name": "get_weather", "arguments": {"city": "Paris"}}\n</tool_call>\n<tool_call>\n{"name": "get_weather", "ar'
  no_tool    calls=[]  reply_tail='2 + 2 = 4'
  pick_tool  calls=[{'name': 'get_time', 'arguments': {'city': 'Tokyo'}}]  reply_tail='<tool_call>\n{"name": "get_time", "arguments": {"city": "Tokyo"}}\n</tool_c

**Prompt mode.** Some chat templates cannot render tools (Tiny Aya, Aya Expanse, Aya Vision). For those, `ToolCallAdapter(mode="prompt")` writes the schemas into a system message and asks for `<tool_call>` blocks. This works for any text model. Here is the same model in prompt mode, for comparison:

In [5]:
r = ak.evaluate(TASKS, model=qwen, adapter=ak.ToolCallAdapter(mode="prompt"), scorers=SCORERS(), config=RunConfig(max_tokens=MAX_TOKENS))
show(r, "hf: prompt-mode tools"); RESULTS["1 hf prompt"] = r.headline

=== hf: prompt-mode tools ===
headline: {'agent_loop_detection': 1.0, 'parallel_detection': 1.0, 'parallel_precision': 1.0, 'parallel_recall': 1.0, 'tool_call_exact': 1.0, 'tool_call_f1': 1.0, 'tool_call_precision': 1.0, 'tool_call_recall': 1.0, 'tool_call_validity': 1.0, 'tool_permission': 1.0, 'trajectory_in_order': 1.0, 'trajectory_strict': 1.0}
  single     calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}}]  reply_tail='{"name": "get_weather", "arguments": {"city": "Paris"}}'
  parallel   calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}}, {'name': 'get_weather', 'arguments': {'city': 'Rome'}}]  reply_tail='<tool_call>{"name": "get_weather", "arguments": {"city": "Paris"}}</tool_call>\n<tool_call>{"name": "get_weather", "argum'
  no_tool    calls=[]  reply_tail='2 + 2 = 4'
  pick_tool  calls=[{'name': 'get_time', 'arguments': {'city': 'Tokyo'}}]  reply_tail='{"name": "get_time", "arguments": {"city": "Tokyo"}}'


## 2 · `hf:`, a multi-turn conversation

Put the conversation so far in `Sample.metadata["messages"]` (OpenAI format, tool results included). The template renders it, and the model continues from there. Here the tool result is already present, so the right move is to **answer, not call again**. `expected_tool_calls=[]` scores that.

In [6]:
from auditkit.registry import METRICS
history = [
    {"role": "user", "content": "What's the weather in Paris?"},
    {"role": "assistant", "content": "", "tool_calls": [{"id": "c1", "type": "function",
        "function": {"name": "get_weather", "arguments": json.dumps({"city": "Paris"})}}]},
    {"role": "tool", "tool_call_id": "c1", "content": json.dumps({"temp_c": 18, "sky": "sunny"})},
]
s = ak.Sample(id="answer_from_result", input="What's the weather in Paris?", tools=TOOLS,
              metadata={"messages": history}, expected_tool_calls=[], target="sunny")
r = ak.evaluate([s], model=qwen, adapter=ak.ToolCallAdapter(),
                scorers=[ak.ToolCallF1(), METRICS.get("contains")(substring="sunny")],
                config=RunConfig(max_tokens=MAX_TOKENS))
show(r, "hf: multi-turn"); RESULTS["2 hf multi-turn"] = r.headline

=== hf: multi-turn ===
headline: {'contains(sunny)': 1.0, 'tool_call_exact': 1.0, 'tool_call_f1': 1.0, 'tool_call_precision': 1.0, 'tool_call_recall': 1.0}
  answer_from_result calls=[]  reply_tail='The current weather in Paris is **18°C** with **sunny** skies. ☀️'


## 3 · Cohere models through `hf:`

**3a · Tiny Aya.** Its chat template does not render tools, so native mode is **refused with advice** rather than silently dropping the tools. Prompt mode is the way to evaluate it. The model is gated, so accept the licence on the Hub and set `HF_TOKEN`.

In [7]:
from auditkit.errors import CapabilityError
if os.environ.get("HF_TOKEN"):
    aya = HFGenModel(TINY_AYA, device=DEVICE, name=f"hf:{TINY_AYA}", token=os.environ["HF_TOKEN"])
    try:
        ak.evaluate(TASKS[:1], model=aya, adapter=ak.ToolCallAdapter(), scorers=[ak.ToolCallF1()])
    except CapabilityError as e:
        print("native mode refused, as expected:\n ", e)
    r = ak.evaluate(TASKS, model=aya, adapter=ak.ToolCallAdapter(mode="prompt"), scorers=SCORERS(),
                    config=RunConfig(max_tokens=MAX_TOKENS))
    show(r, "Tiny Aya, prompt mode"); RESULTS["3a tiny aya prompt"] = r.headline
    del aya; torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("skipped: set HF_TOKEN (and accept the Tiny Aya licence)")

skipped: set HF_TOKEN (and accept the Tiny Aya licence)


**3b · Command R7B, native tools.** Its template renders tools, and it replies in Cohere's own format, `<|START_ACTION|>[{"tool_name": ..., "parameters": ...}]<|END_ACTION|>`. The markers are special tokens and get stripped on decode; AuditKIT reads the action list either way. It needs a 24 GB+ GPU.

In [8]:
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
if os.environ.get("HF_TOKEN") and mem >= 23:
    r7b = HFGenModel(COMMAND_R7B, device=DEVICE, name=f"hf:{COMMAND_R7B}", token=os.environ["HF_TOKEN"])
    r = ak.evaluate(TASKS, model=r7b, adapter=ak.ToolCallAdapter(), scorers=SCORERS(), config=RunConfig(max_tokens=MAX_TOKENS))
    show(r, "Command R7B, native"); RESULTS["3b command r7b native"] = r.headline
    del r7b; torch.cuda.empty_cache()
else:
    print(f"skipped: needs HF_TOKEN and a 24 GB+ GPU (this one: {mem:.0f} GB)")

skipped: needs HF_TOKEN and a 24 GB+ GPU (this one: 0 GB)


## 4 · Cohere's hosted API through `api:`

`api:` speaks the OpenAI chat API, and Cohere serves one at `https://api.cohere.ai/compatibility/v1`. Tool calls come back **structured** (`message.tool_calls`), so there is no text to parse. Set `CO_API_KEY`.

In [9]:
if os.environ.get("CO_API_KEY"):
    from auditkit.model.api_gen import APIModel
    cohere_api = APIModel(COHERE_API_MODEL, api_base="https://api.cohere.ai/compatibility/v1",
                          api_key=os.environ["CO_API_KEY"], name=f"api:cohere/{COHERE_API_MODEL}")
    r = ak.evaluate(TASKS, model=cohere_api, adapter=ak.ToolCallAdapter(), scorers=SCORERS(), config=RunConfig(max_tokens=512))
    show(r, "Cohere API"); RESULTS["4 cohere api"] = r.headline
else:
    print("skipped: set CO_API_KEY")

skipped: set CO_API_KEY


## 5 · A deployed agent behind `agent:`

A deployed agent (an AgentTune-served agent, a LangGraph app, …) runs its **own** tool loop and returns the finished result. `agent:<url>` POSTs `{"input": ...}` and reads back:
- `output`, the final answer;
- `messages`, the OpenAI-format transcript including tool calls and results;
- `contexts` (optional), retrieved documents.

Below, a small agent server runs the **real** Qwen model in a real loop and executes real (toy) tools. AuditKIT then scores the transcript it returns.

In [10]:
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from auditkit.model import Request
from auditkit.trace import parse_tool_calls

WEATHER = {"Paris": "sunny, 18C", "Rome": "cloudy, 22C", "Tokyo": "rain, 15C"}
def run_tool(name, args):
    city = args.get("city", "")
    return WEATHER.get(city, "unknown city") if name == "get_weather" else f"14:05 in {city}"

def agent_loop(user_input, max_steps=4):
    """The agent's own loop: model -> tool calls -> tool results -> model ... -> answer."""
    messages = [{"role": "user", "content": user_input}]
    for step in range(max_steps):
        out = qwen.generate([Request(prompt=user_input, request_type="chat",
                                     params={"messages": messages, "tools": TOOLS, "max_tokens": MAX_TOKENS})])
        text = out[0].completions[0].text or ""
        calls = parse_tool_calls(text)
        if not calls:
            messages.append({"role": "assistant", "content": text})
            return {"output": text.split("</think>")[-1].strip(), "messages": messages}
        ids = [f"s{step}c{i}" for i in range(len(calls))]
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"id": i, "type": "function", "function": {"name": c.name, "arguments": json.dumps(c.arguments)}}
            for i, c in zip(ids, calls)]})
        for i, c in zip(ids, calls):
            messages.append({"role": "tool", "tool_call_id": i, "content": run_tool(c.name, c.arguments)})
    return {"output": "", "messages": messages}

class AgentHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        data = json.dumps(agent_loop(body["input"])).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *a): pass

server = ThreadingHTTPServer(("127.0.0.1", 0), AgentHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
AGENT_URL = f"http://127.0.0.1:{server.server_address[1]}/run"
print("agent listening at", AGENT_URL)

agent listening at http://127.0.0.1:PORT/run


In [11]:
# The agent keeps its tools server-side, so the samples carry only the task and the reference calls.
agent_tasks = [ak.Sample(id=s.id, input=s.input, expected_tool_calls=s.expected_tool_calls) for s in TASKS]
r = ak.evaluate(agent_tasks, model=f"agent:{AGENT_URL}", scorers=[ak.ToolCallF1(), ak.TrajectoryMatch(),
                ak.ParallelToolCalls(), ak.AgentLoopDetection()], config=RunConfig(timeout=600))
show(r, "agent: endpoint"); RESULTS["5 agent endpoint"] = r.headline
for p in r.predictions:
    print(" ", p.sample_id, "-> final answer:", (p.raw_output or "")[:100])

=== agent: endpoint ===
headline: {'agent_loop_detection': 1.0, 'parallel_detection': 1.0, 'parallel_precision': 1.0, 'parallel_recall': 1.0, 'tool_call_exact': 1.0, 'tool_call_f1': 1.0, 'tool_call_precision': 1.0, 'tool_call_recall': 1.0, 'trajectory_in_order': 1.0, 'trajectory_strict': 1.0}
  single     calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}, 'id': 's0c0'}]  reply_tail='The weather in Paris is sunny with a temperature of 18°C. 🌞'
  parallel   calls=[{'name': 'get_weather', 'arguments': {'city': 'Paris'}, 'id': 's0c0'}, {'name': 'get_weather', 'arguments': {'city': 'Rome'}, 'id': 's0c1'}]  reply_tail="The weather in Paris is **sunny** with a temperature of **18°C**. In Rome, it's **cloudy** with a temperature of **22°C*"
  no_tool    calls=[]  reply_tail='2 + 2 = 4'
  pick_tool  calls=[{'name': 'get_time', 'arguments': {'city': 'Tokyo'}, 'id': 's0c0'}]  reply_tail='The current local time in Tokyo is 14:05.'
  single -> final answer: The weather in Paris is sunny

## 6 · `agent_eval` harness: AuditKIT runs the loop and verifies the outcome

Here AuditKIT **owns** the loop. It sends the task to the policy model, executes each tool call against a **tool environment you supply** (a test double, never real side effects), feeds the results back, stops on an answer or at `max_steps`, and then checks the **final state** with an oracle. The verdict comes from what actually happened to the state, not from what the reply says.

> Note: today the harness reads tool calls only from a structured `trace`, as `api:` and `agent:` return them, and it passes no generation settings. For a text model (`hf:`, `vllm:`, Cohere), wrap it as below so its text calls are parsed and it gets enough tokens. Without the wrapper, the first call is mistaken for the final answer (reported to the maintainers).

In [12]:
from auditkit.agent_eval import AgentCase, AgentEvalRunner, AgentEvalSpec, FinalStateAssertion
from auditkit.model import Model

class TextToolCalls(Model):
    """Makes a text-output model usable as a harness policy: parses its tool calls into the trace."""
    def __init__(self, inner, max_tokens=1536):
        self.inner, self.name, self.max_tokens = inner, inner.name, max_tokens
    def capabilities(self):
        return self.inner.capabilities()
    def generate(self, requests):
        for r in requests:
            r.params.setdefault("max_tokens", self.max_tokens)
        out = self.inner.generate(requests)
        for res in out:
            g = res.completions[0]
            calls = parse_tool_calls(g.text or "")
            if calls and not (g.trace or {}).get("tool_calls"):
                g.trace = {**(g.trace or {}), "tool_calls": [[{"name": c.name, "arguments": c.arguments} for c in calls]]}
        return out

BANK_TOOLS = [
    {"type": "function", "function": {"name": "transfer", "description": "Move money between the user's accounts",
     "parameters": {"type": "object", "properties": {"src": {"type": "string"}, "dst": {"type": "string"},
                    "amount": {"type": "number"}}, "required": ["src", "dst", "amount"]}}},
    {"type": "function", "function": {"name": "get_balance", "description": "Balance of one account",
     "parameters": {"type": "object", "properties": {"account": {"type": "string"}}, "required": ["account"]}}},
]

class Bank:
    """The tool environment: a resettable double whose snapshot() the oracle checks."""
    def __init__(self):
        self.acc = {"checking": 1000.0, "savings": 0.0}
    def __call__(self, name, args):
        if name == "get_balance":
            return json.dumps({"balance": self.acc.get(args.get("account"))})
        if name == "transfer":
            amount = float(args["amount"])
            self.acc[args["src"]] -= amount
            self.acc[args["dst"]] += amount
            return "ok"
        return "unknown tool"
    def snapshot(self):
        return dict(self.acc)

case = AgentCase(id="move-250", task="Move 250 from checking to savings, then tell me the new savings balance.",
                 allowed_tools=BANK_TOOLS, outcome=FinalStateAssertion("savings", equals=250.0))
res = AgentEvalRunner().run(AgentEvalSpec(
    cases=[case], mode="harness", agent=TextToolCalls(qwen),
    agent_opts={"tool_env": Bank(), "max_steps": 5},
    scorers=["tool_call_validity", "agent_loop_detection"]))
row, ep = res.rows[0], res.episodes[0]
print("status:", row.status, "| stop:", row.stop_reason, "| outcome:", row.outcome["verdict"])
print("verified final state:", ep.final_state)
print("calls made:", [(e.payload.get("function") or e.payload).get("name") for e in ep.tool_call_events()])
print("answer:", (ep.final_answer or "").split("</think>")[-1].strip()[:160])
RESULTS["6 harness outcome"] = {"success": float(row.outcome["verdict"] == "success")}

status: completed | stop: stop | outcome: success
verified final state: {'checking': 750.0, 'savings': 250.0}
calls made: ['transfer', 'get_balance']
answer: The new savings balance after transferring $250 is **$250.00**.


## Summary

In [13]:
import math
for name, h in RESULTS.items():
    keys = [k for k in ("tool_call_f1", "trajectory_exact", "parallel_detection", "tool_call_validity",
                        "tool_permission", "agent_loop_detection", "contains", "success") if k in h]
    print(f"{name:24s}", {k: round(h[k], 3) for k in keys})

1 hf native              {'tool_call_f1': 1.0, 'parallel_detection': 1.0, 'tool_call_validity': 1.0, 'tool_permission': 1.0, 'agent_loop_detection': 1.0}
1 hf prompt              {'tool_call_f1': 1.0, 'parallel_detection': 1.0, 'tool_call_validity': 1.0, 'tool_permission': 1.0, 'agent_loop_detection': 1.0}
2 hf multi-turn          {'tool_call_f1': 1.0}
5 agent endpoint         {'tool_call_f1': 1.0, 'parallel_detection': 1.0, 'agent_loop_detection': 1.0}
6 harness outcome        {'success': 1.0}


### Reading the numbers
- **`tool_call_f1`**: the right calls with the right arguments, matched as a set.
- **`trajectory_*`**: the calls in the right order.
- **`parallel_*`**: calls that should be made together are made together.
- **`tool_call_validity`**: the calls conform to the tool schemas.
- **`tool_permission`**: only offered or allowed tools were called.
- **`agent_loop_detection`**: 1.0 means no loop.
- **Irrelevance** (`no_tool`) passes when the model answers without calling.
- **Harness `outcome`**: whether the verified environment state matches the case's oracle.